In [3]:
"""
model.py – Trains and saves the DeepShot ensemble model bundle.

Upgrade Summary
---------------
Phase 2  Validation & Pre-Processing
    2.1  TimeSeriesSplit(n_splits=5) throughout.
    2.2  RobustScaler applied before all models; stored in bundle.

Phase 3  Model Optimisation (Optuna → RFECV order)
    3.1  Optuna runs FIRST on the full feature set for each base model.
         Best XGBoost params are then used as the RFECV estimator so that
         feature importance scores reflect an already-tuned model.
    3.2  n_trials=100 per Optuna study.
    3.3  Separate Optuna studies for XGBoost, LightGBM, and CatBoost.

Phase 4  Ensemble Architecture
    4.1  LGBMClassifier and CatBoostClassifier added.
    4.2  VotingClassifier(voting='soft') with log-loss-derived weights.
    4.3  StackingClassifier (LogisticRegression meta-learner) as optional
         advanced alternative.

Phase 5  Evaluation & Calibration
    5.1  CalibratedClassifierCV(method='isotonic', cv=TimeSeriesSplit)
         applied to the final voting ensemble.
    5.2  ROI & Edge evaluation: The Edge, ROI @ >3% edge, Brier Score.
    5.3  Optimal-threshold logic removed; model outputs raw probabilities.

Fix (Phase 4.3 / 5.1)
    StackingClassifier.cv uses KFold(n_splits=3, shuffle=False) instead of
    TimeSeriesSplit.  TimeSeriesSplit is incompatible with cross_val_predict
    (used internally by StackingClassifier) because early samples never
    appear in any test fold, violating the partition requirement.
    CalibratedClassifierCV wrapping the stacking model also uses
    KFold(n_splits=3, shuffle=False) for the same reason.

Enhancement: Dynamic Feature Alignment
    Added feature alignment utilities for inference to handle:
    - Missing features (filled with 0)
    - Extra features (ignored)
    - Feature order consistency
"""

from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import joblib
import optuna
import warnings

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

MODEL_OUTPUT_PATH = "deepshot.pkl"

# ── Tuneable constants ────────────────────────────────────────────────────────
N_SPLITS = 5          # Phase 2.1
N_OPTUNA_TRIALS = 100  # Phase 3.2
EDGE_THRESHOLD = 0.03  # Phase 5.2  – minimum model edge to count as value bet
MIN_FEATURES = 15      # RFECV minimum


# ── Feature Engineering ───────────────────────────────────────────────────────

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parse raw Vegas moneylines into implied probabilities and derive
    market-relative features.  Must be kept in sync with main.py.

    New features produced
    ---------------------
    home_moneyline_implied_prob, away_moneyline_implied_prob
    prob_diff, prob_ratio
    elo_diff, elo_ratio  (if Elo columns present)
    """
    df = df.copy()
    
    for col in ("home_moneyline", "away_moneyline"):
        if col not in df.columns:
            continue
        clean_str = (
            df[col]
            .astype(str)
            .str.replace("+", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.replace(" ", "", regex=False)
        )
        numeric_odds = pd.to_numeric(clean_str, errors="coerce")

        implied_prob = np.where(
            numeric_odds >= 100,
            100 / (numeric_odds + 100),
            np.where(
                numeric_odds <= -100,
                np.abs(numeric_odds) / (np.abs(numeric_odds) + 100),
                np.where(
                    (numeric_odds > 1.0) & (numeric_odds < 100.0),
                    1.0 / numeric_odds,
                    np.nan,
                ),
            ),
        )
        df[f"{col}_implied_prob"] = implied_prob

    req = ("home_moneyline_implied_prob", "away_moneyline_implied_prob")
    if all(c in df.columns for c in req):
        df["prob_diff"] = df[req[0]] - df[req[1]]
        df["prob_ratio"] = df[req[0]] / (df[req[1]] + 1e-5)

    if "home_elo" in df.columns and "away_elo" in df.columns:
        df["elo_diff"] = df["home_elo"] - df["away_elo"]
        df["elo_ratio"] = df["home_elo"] / (df["away_elo"] + 1e-5)

    return df


# ── Dynamic Feature Alignment Utilities ──────────────────────────────────────

def align_features_for_inference(
    df: pd.DataFrame,
    required_features: List[str],
    scaler: Optional[RobustScaler] = None,
    fill_value: float = 0.0
) -> pd.DataFrame:
    """
    Align features for inference to match training features.
    
    Handles:
    - Missing features (filled with fill_value)
    - Extra features (ignored)
    - Feature order consistency
    - Optional scaling
    
    Parameters
    ----------
    df : pd.DataFrame
        Input data for inference
    required_features : List[str]
        Features required by the model (from bundle['features'])
    scaler : RobustScaler, optional
        Scaler to apply if provided (from bundle['scaler'])
    fill_value : float
        Value to fill missing features with
        
    Returns
    -------
    pd.DataFrame
        Aligned and optionally scaled features
    """
    df_aligned = df.copy()
    
    # Add missing features with fill_value
    missing_features = [f for f in required_features if f not in df_aligned.columns]
    if missing_features:
        print(f"Warning: Adding {len(missing_features)} missing features: {missing_features}")
        for feat in missing_features:
            df_aligned[feat] = fill_value
    
    # Select only required features in correct order
    df_aligned = df_aligned[required_features]
    
    # Apply scaling if provided
    if scaler is not None:
        df_aligned = pd.DataFrame(
            scaler.transform(df_aligned),
            columns=df_aligned.columns,
            index=df_aligned.index
        )
    
    return df_aligned


def validate_inference_data(
    df: pd.DataFrame,
    required_features: List[str],
    required_columns: Optional[List[str]] = None
) -> Tuple[bool, str]:
    """
    Validate inference data has necessary columns and structure.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input data for inference
    required_features : List[str]
        Features required by the model
    required_columns : List[str], optional
        Additional required columns (e.g., for feature engineering)
        
    Returns
    -------
    Tuple[bool, str]
        (is_valid, error_message)
    """
    if df.empty:
        return False, "Input DataFrame is empty"
    
    # Check for required columns for feature engineering
    if required_columns:
        missing_cols = [col for col in required_columns if col not in df.columns]
        if missing_cols:
            return False, f"Missing required columns for feature engineering: {missing_cols}"
    
    # Check for NaN values in critical columns
    if required_columns:
        nan_counts = df[required_columns].isna().sum()
        if nan_counts.any():
            return False, f"NaN values found in required columns: {nan_counts[nan_counts > 0].to_dict()}"
    
    return True, "Validation passed"


# ── Optuna Objectives ─────────────────────────────────────────────────────────

def _make_xgb_objective(X: pd.DataFrame, y: pd.Series, tscv: TimeSeriesSplit):
    """Return an Optuna objective for XGBoost."""

    def objective(trial: optuna.Trial) -> float:
        params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.15, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 6.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 25),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "random_state": 42,
            "n_jobs": -1,
        }
        cv_scores = []
        for train_idx, val_idx in tscv.split(X):
            X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
            y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
            mdl = XGBClassifier(**params)
            mdl.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
            cv_scores.append(log_loss(y_v, mdl.predict_proba(X_v)[:, 1]))
        return float(np.mean(cv_scores))

    return objective


def _make_lgbm_objective(X: pd.DataFrame, y: pd.Series, tscv: TimeSeriesSplit):
    """Return an Optuna objective for LightGBM."""

    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 20, 200),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "random_state": 42,
            "n_jobs": -1,
            "verbose": -1,
        }
        cv_scores = []
        for train_idx, val_idx in tscv.split(X):
            X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
            y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
            mdl = LGBMClassifier(**params)
            mdl.fit(X_t, y_t)
            cv_scores.append(log_loss(y_v, mdl.predict_proba(X_v)[:, 1]))
        return float(np.mean(cv_scores))

    return objective


def _make_catboost_objective(X: pd.DataFrame, y: pd.Series, tscv: TimeSeriesSplit):
    """Return an Optuna objective for CatBoost."""

    def objective(trial: optuna.Trial) -> float:
        params = {
            "iterations": trial.suggest_int("iterations", 100, 800),
            "depth": trial.suggest_int("depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.15, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 20.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "random_seed": 42,
            "verbose": 0,
            "allow_writing_files": False,
        }
        cv_scores = []
        for train_idx, val_idx in tscv.split(X):
            X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
            y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
            mdl = CatBoostClassifier(**params)
            mdl.fit(X_t, y_t, verbose=0)
            cv_scores.append(log_loss(y_v, mdl.predict_proba(X_v)[:, 1]))
        return float(np.mean(cv_scores))

    return objective


def _run_optuna_study(
    name: str,
    objective,
    n_trials: int,
) -> optuna.Study:
    """Run a minimisation study and return it."""
    print(f"\n{'─'*55}")
    print(f"  Optuna Study: {name}  ({n_trials} trials)")
    print(f"{'─'*55}")
    study = optuna.create_study(
        direction="minimize",
        study_name=name,
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(
        f"  ✓ Best log-loss: {study.best_value:.5f} "
        f"(trial #{study.best_trial.number})"
    )
    return study


# ── ROI / Edge Helpers ────────────────────────────────────────────────────────

def _calculate_roi(
    test_df: pd.DataFrame,
    y_pred_proba: np.ndarray,
    edge_threshold: float = EDGE_THRESHOLD,
) -> None:
    """
    Phase 5.2 – Compute and print Edge, ROI, and Brier Score metrics.

    Metric 1 – The Edge
        edge = model_prob_home − implied_vegas_prob_home
        A positive edge means the model believes the home team is a better
        bet than the market implies.

    Metric 2 – ROI Analysis
        Flat $1 unit bet on all games where edge > ``edge_threshold``.
        Payout computed from the implied probability (decimal odds = 1/p).
        ROI = (net_profit / total_wagered) × 100 %

    Metric 3 – Brier Score
        Primary anchor for probability accuracy (lower = better).
    """
    df = test_df.copy()
    df["model_prob"] = y_pred_proba

    brier = brier_score_loss(df["winning_team"], y_pred_proba)
    roc_auc = roc_auc_score(df["winning_team"], y_pred_proba)
    logloss = log_loss(df["winning_team"], y_pred_proba)

    print("\n" + "=" * 55)
    print("     Primary Evaluation Metrics (Phase 5.2 / 5.3)")
    print("=" * 55)
    print(f"  Brier Score (↓ better) : {brier:.5f}")
    print(f"  ROC-AUC                : {roc_auc:.5f}")
    print(f"  Log Loss               : {logloss:.5f}")

    if "home_moneyline_implied_prob" not in df.columns:
        print("\n  [Edge/ROI skipped – no implied prob column in test set]")
        return

    df["edge"] = df["model_prob"] - df["home_moneyline_implied_prob"]

    print("\n" + "─" * 55)
    print("  The Edge  (model_prob − vegas_implied_prob)")
    print("─" * 55)
    pos_edge = df[df["edge"] > 0]
    neg_edge = df[df["edge"] <= 0]
    print(f"  Games with positive edge : {len(pos_edge)}")
    print(f"  Games with negative edge : {len(neg_edge)}")
    if not df.empty:
        print(f"  Mean edge (all games)    : {df['edge'].mean():+.4f}")

    value_bets = df[df["edge"] > edge_threshold].copy()
    print(f"\n  Value bets (edge > {edge_threshold:.0%}) : {len(value_bets)}")

    if value_bets.empty:
        print("  [No value bets found at this threshold]")
        return

    def flat_bet_profit(row: pd.Series) -> float:
        """$1 flat bet profit when model has edge on home team winning."""
        implied_prob = row["home_moneyline_implied_prob"]
        if implied_prob <= 0 or pd.isna(implied_prob):
            return -1.0
        decimal_odds = 1.0 / implied_prob
        return (decimal_odds - 1.0) if row["winning_team"] == 1 else -1.0

    value_bets["profit"] = value_bets.apply(flat_bet_profit, axis=1)
    total_wagered = float(len(value_bets))
    net_profit = value_bets["profit"].sum()
    roi = (net_profit / total_wagered) * 100.0
    win_rate = (value_bets["winning_team"] == 1).mean()
    avg_edge = value_bets["edge"].mean()

    print("\n" + "─" * 55)
    print(f"  ROI Analysis  (flat $1 unit on edge > {edge_threshold:.0%})")
    print("─" * 55)
    print(f"  Win rate on value bets  : {win_rate:.2%}")
    print(f"  Average edge            : {avg_edge:+.4f}")
    print(f"  Net profit (units)      : {net_profit:+.2f}")
    print(f"  ROI                     : {roi:+.2f}%")

    # Calibration bins
    print("\n" + "─" * 55)
    print("  Probability Calibration Bins")
    print("─" * 55)
    bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    for lo, hi in zip(bins, bins[1:]):
        mask = (y_pred_proba >= lo) & (y_pred_proba < hi)
        if mask.sum() > 0:
            # FIXED: Use mask directly instead of mask.values
            actual_rate = float(df.loc[mask, "winning_team"].mean())
            avg_pred = float(y_pred_proba[mask].mean())
            print(
                f"  {lo:.1f}–{hi:.1f} | pred {avg_pred:.1%} "
                f"| actual {actual_rate:.1%} | n={mask.sum()}"
            )


# ═════════════════════════════════════════════════════════════════════════════
# Main Training Pipeline
# ═════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    Path(MODEL_OUTPUT_PATH).unlink(missing_ok=True)

    # ── Load & Prepare Data ───────────────────────────────────────────────────
    print("Loading Data...")
    df = pd.read_csv("../data/csv/dataset.csv")
    
    # Validate required columns exist
    required_raw_cols = ["date", "winning_team", "home_moneyline", "away_moneyline"]
    missing_cols = [col for col in required_raw_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in dataset: {missing_cols}")
    
    df = feature_engineering(df)

    req_odds = ("home_moneyline_implied_prob", "away_moneyline_implied_prob")
    if all(c in df.columns for c in req_odds):
        initial_len = len(df)
        df = df.dropna(subset=list(req_odds)).reset_index(drop=True)
        dropped = initial_len - len(df)
        if dropped:
            print(
                f"Dropped {dropped} games missing Vegas odds "
                "(forcing correct training priorities)."
            )

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)

    # ── Chronological Train / Test Split (85 / 15) ────────────────────────────
    print("\nSplitting the dataset chronologically (85% Train / 15% Test)...")
    split_idx = int(len(df) * 0.85)
    train_full_df = df.iloc[:split_idx].reset_index(drop=True)
    test_df = df.iloc[split_idx:].reset_index(drop=True)

    if len(test_df) == 0:
        raise ValueError("Test set is empty!")

    COLS_TO_DROP = [
        "winning_team", "date", "home_team", "away_team",
        "home_moneyline", "away_moneyline",
    ]

    X_train_full = train_full_df.drop(
        columns=[c for c in COLS_TO_DROP if c in train_full_df.columns]
    )
    y_train_full = train_full_df["winning_team"]

    X_test = test_df.drop(
        columns=[c for c in COLS_TO_DROP if c in test_df.columns]
    )
    y_test = test_df["winning_team"]

    # Remove any columns that are entirely NaN or non-numeric
    X_train_full = X_train_full.select_dtypes(include=[np.number]).fillna(0)
    X_test = X_test[X_train_full.columns].fillna(0)

    # ── Phase 2.1: TimeSeriesSplit(n_splits=5) ────────────────────────────────
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    print(f"\nUsing TimeSeriesSplit(n_splits={N_SPLITS})  [Phase 2.1]")

    # ── Phase 2.2: RobustScaler ───────────────────────────────────────────────
    print("Fitting RobustScaler...  [Phase 2.2]")
    scaler = RobustScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_full),
        columns=X_train_full.columns,
        index=X_train_full.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )
    print(f"  Starting features : {X_train_scaled.shape[1]}")

    # ═════════════════════════════════════════════════════════════════════════
    # Phase 3.3 – Multi-Model Optuna Studies on Full Feature Set
    # Phase 3.1 – Optuna FIRST, then RFECV with tuned estimator
    # Phase 3.2 – 100 trials per study
    # ═════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 55)
    print("  Phase 3: Multi-Model Optuna Optimisation")
    print("=" * 55)
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # --- XGBoost ---
    xgb_study = _run_optuna_study(
        name="xgboost",
        objective=_make_xgb_objective(X_train_scaled, y_train_full, tscv),
        n_trials=N_OPTUNA_TRIALS,
    )
    best_xgb_params = {
        **xgb_study.best_params,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }

    # --- LightGBM ---
    lgbm_study = _run_optuna_study(
        name="lightgbm",
        objective=_make_lgbm_objective(X_train_scaled, y_train_full, tscv),
        n_trials=N_OPTUNA_TRIALS,
    )
    best_lgbm_params = {
        **lgbm_study.best_params,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    # --- CatBoost ---
    catboost_study = _run_optuna_study(
        name="catboost",
        objective=_make_catboost_objective(X_train_scaled, y_train_full, tscv),
        n_trials=N_OPTUNA_TRIALS,
    )
    best_catboost_params = {
        **catboost_study.best_params,
        "random_seed": 42,
        "verbose": 0,
        "allow_writing_files": False,
    }

    # ═════════════════════════════════════════════════════════════════════════
    # Phase 3.1 – RFECV Using Tuned XGBoost (feature importance more accurate
    #             when the model is already optimised)
    # ═════════════════════════════════════════════════════════════════════════
    print(
        f"\n{'='*55}\n"
        f"  Phase 3.1: RFECV with Tuned XGBoost\n"
        f"  (starting from {X_train_scaled.shape[1]} features)\n"
        f"{'='*55}"
    )
    tuned_xgb_for_rfecv = XGBClassifier(**best_xgb_params)
    selector = RFECV(
        estimator=tuned_xgb_for_rfecv,
        step=1,
        cv=tscv,
        scoring="neg_log_loss",
        min_features_to_select=MIN_FEATURES,
        n_jobs=-1,
    )
    selector.fit(X_train_scaled, y_train_full)

    selected_features = X_train_scaled.columns[selector.support_].tolist()
    X_train = X_train_scaled[selected_features]
    X_test_final = X_test_scaled[selected_features]

    print(
        f"\nRFECV complete!  "
        f"{len(selected_features)} features selected from "
        f"{X_train_scaled.shape[1]} candidates."
    )
    print("Selected features:")
    for feat in selected_features:
        print(f"  • {feat}")

    # ═════════════════════════════════════════════════════════════════════════
    # Phase 4 – Ensemble Architecture
    # ═════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 55)
    print("  Phase 4: Evaluating Base Models for Ensemble Weights")
    print("=" * 55)

    base_models = {
        "xgboost": XGBClassifier(**best_xgb_params),
        "lightgbm": LGBMClassifier(**best_lgbm_params),
        "catboost": CatBoostClassifier(**best_catboost_params),
    }

    model_log_losses: dict = {}
    for name, mdl in base_models.items():
        scores = cross_val_score(
            mdl,
            X_train,
            y_train_full,
            cv=tscv,
            scoring="neg_log_loss",
            n_jobs=-1,
        )
        mean_ll = float(-np.mean(scores))
        model_log_losses[name] = mean_ll
        print(f"  {name:<12} CV log-loss: {mean_ll:.5f}")

    # Phase 4.2 – Soft voting weights: inverse log-loss (lower loss → higher weight)
    inv_losses = {k: 1.0 / v for k, v in model_log_losses.items()}
    total_inv = sum(inv_losses.values())
    weights = [round(inv_losses[n] / total_inv, 4) for n in base_models]

    print(f"\n  Voting weights (log-loss-derived): "
          f"{dict(zip(base_models, weights))}")

    estimators = list(base_models.items())

    # Phase 4.2 – VotingClassifier (soft)
    voting_clf = VotingClassifier(
        estimators=estimators,
        voting="soft",
        weights=weights,
        n_jobs=-1,
    )

    # Phase 4.3 – StackingClassifier (advanced, optional)
    # FIX: Uses KFold(shuffle=False) instead of TimeSeriesSplit.
    # TimeSeriesSplit is incompatible with cross_val_predict (used internally
    # by StackingClassifier) because early samples never appear in any test
    # fold, violating sklearn's partition requirement.
    # KFold(shuffle=False) preserves temporal order while satisfying the
    # partition constraint. This is the approach recommended by the sklearn
    # core team. See: https://github.com/scikit-learn/scikit-learn/issues/17285
    stacking_clf = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(
            C=1.0,
            max_iter=500,
            random_state=42,
        ),
        cv=KFold(n_splits=3, shuffle=False),  # ← FIXED: was TimeSeriesSplit(n_splits=3)
        stack_method="predict_proba",
        n_jobs=-1,
    )

    # ═════════════════════════════════════════════════════════════════════════
    # Phase 5.1 – Isotonic Calibration Wrapped in TimeSeriesSplit
    # ═════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 55)
    print("  Phase 5.1: Calibrating Ensemble (isotonic / TimeSeriesSplit)")
    print("=" * 55)

    tscv_calib = TimeSeriesSplit(n_splits=3)

    # Primary: calibrated VotingClassifier
    calibrated_voting = CalibratedClassifierCV(
        estimator=voting_clf,
        method="isotonic",
        cv=tscv_calib,
    )
    print("  Fitting calibrated VotingClassifier...")
    calibrated_voting.fit(X_train, y_train_full)

    # Alternative: calibrated StackingClassifier
    # FIX: Uses KFold(shuffle=False) for the same partition-compatibility reason.
    calibrated_stacking = CalibratedClassifierCV(
        estimator=stacking_clf,
        method="isotonic",
        cv=KFold(n_splits=3, shuffle=False),  # ← FIXED: was tscv_calib
    )
    print("  Fitting calibrated StackingClassifier...")
    calibrated_stacking.fit(X_train, y_train_full)

    # ─── Choose best model by CV log-loss ─────────────────────────────────────
    voting_scores = cross_val_score(
        calibrated_voting,
        X_train,
        y_train_full,
        cv=TimeSeriesSplit(n_splits=3),
        scoring="neg_log_loss",
    )
    stacking_scores = cross_val_score(
        calibrated_stacking,
        X_train,
        y_train_full,
        cv=TimeSeriesSplit(n_splits=3),
        scoring="neg_log_loss",
    )

    voting_mean_ll = -float(np.mean(voting_scores))
    stacking_mean_ll = -float(np.mean(stacking_scores))

    print(f"\n  Voting   CV log-loss: {voting_mean_ll:.5f}")
    print(f"  Stacking CV log-loss: {stacking_mean_ll:.5f}")

    if stacking_mean_ll < voting_mean_ll:
        final_model = calibrated_stacking
        chosen_arch = "StackingClassifier"
    else:
        final_model = calibrated_voting
        chosen_arch = "VotingClassifier"

    print(f"\n  ✓ Selected architecture: {chosen_arch}")

    # ═════════════════════════════════════════════════════════════════════════
    # Phase 5.2 & 5.3 – Evaluation (Edge / ROI / Brier; NO threshold logic)
    # ═════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 55)
    print("  Phase 5.2: ROI & Edge Evaluation on Test Set")
    print("=" * 55)

    y_pred_proba = final_model.predict_proba(X_test_final)[:, 1]
    _calculate_roi(test_df, y_pred_proba, edge_threshold=EDGE_THRESHOLD)

    # ═════════════════════════════════════════════════════════════════════════
    # Save Model Bundle
    # ═════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 55)
    print("  Saving Model Bundle")
    print("=" * 55)

    brier = brier_score_loss(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    logloss = log_loss(y_test, y_pred_proba)

    model_bundle = {
        # Core inference artefacts
        "model": final_model,
        "scaler": scaler,                    # Phase 2.2
        "features": selected_features,

        # Architecture metadata
        "ensemble_architecture": chosen_arch,
        "base_model_weights": dict(zip(base_models, weights)),

        # Best hyperparameters (all three models)
        "best_params": {
            "xgboost": best_xgb_params,
            "lightgbm": best_lgbm_params,
            "catboost": best_catboost_params,
        },

        # Evaluation metrics
        "metrics": {
            "brier_score": round(brier, 6),   # Phase 5.2: primary anchor
            "roc_auc": round(roc_auc, 6),
            "log_loss": round(logloss, 6),
            # Phase 5.3: no optimal_threshold stored
        },

        # Betting-relevant metadata
        "edge_threshold": EDGE_THRESHOLD,    # Phase 5.2
        
        # Feature engineering requirements
        "feature_engineering_required": True,
        "required_columns_for_inference": [
            "home_moneyline", "away_moneyline", "home_elo", "away_elo"
        ],
        
        # Feature importance tracking
        "feature_importance": {
            "rfecv_selected": selected_features,
            "rfecv_ranking": selector.ranking_.tolist(),
            "rfecv_scores": selector.cv_results_["mean_test_score"].tolist(),
        },
        
        # Dynamic alignment utilities
        "alignment_utils": {
            "align_features_for_inference": align_features_for_inference,
            "validate_inference_data": validate_inference_data,
        },
    }

    joblib.dump(model_bundle, MODEL_OUTPUT_PATH)

    print(f"\n  Brier Score : {brier:.5f}  (primary metric)")
    print(f"  ROC-AUC     : {roc_auc:.5f}")
    print(f"  Log Loss    : {logloss:.5f}")
    print(f"\n  Architecture : {chosen_arch}")
    print(f"  Features     : {len(selected_features)}")
    print(f"\n  ✅  Bundle saved to '{MODEL_OUTPUT_PATH}'")
    print(
        "      main.py should load the scaler from bundle['scaler']\n"
        "      and transform inference features before predict_proba()."
    )
    
    # Print usage example for inference
    print("\n" + "=" * 55)
    print("  Inference Usage Example")
    print("=" * 55)
    print("""
    # Load the bundle
    bundle = joblib.load('deepshot.pkl')
    
    # Load and prepare inference data
    inference_df = pd.read_csv('inference_data.csv')
    
    # Validate data
    is_valid, msg = bundle['alignment_utils']['validate_inference_data'](
        inference_df,
        bundle['features'],
        bundle['required_columns_for_inference']
    )
    if not is_valid:
        raise ValueError(f"Data validation failed: {msg}")
    
    # Apply feature engineering
    inference_df = feature_engineering(inference_df)
    
    # Align features
    X_inference = bundle['alignment_utils']['align_features_for_inference'](
        inference_df,
        bundle['features'],
        bundle['scaler']
    )
    
    # Make predictions
    predictions = bundle['model'].predict_proba(X_inference)[:, 1]
    """)


Loading Data...
Dropped 4879 games missing Vegas odds (forcing correct training priorities).

Splitting the dataset chronologically (85% Train / 15% Test)...

Using TimeSeriesSplit(n_splits=5)  [Phase 2.1]
Fitting RobustScaler...  [Phase 2.2]
  Starting features : 113

  Phase 3: Multi-Model Optuna Optimisation

───────────────────────────────────────────────────────
  Optuna Study: xgboost  (100 trials)
───────────────────────────────────────────────────────


  0%|          | 0/100 [00:00<?, ?it/s]

  ✓ Best log-loss: 0.59900 (trial #52)

───────────────────────────────────────────────────────
  Optuna Study: lightgbm  (100 trials)
───────────────────────────────────────────────────────


  0%|          | 0/100 [00:00<?, ?it/s]

  ✓ Best log-loss: 0.60105 (trial #92)

───────────────────────────────────────────────────────
  Optuna Study: catboost  (100 trials)
───────────────────────────────────────────────────────


  0%|          | 0/100 [00:00<?, ?it/s]

  ✓ Best log-loss: 0.59865 (trial #69)

  Phase 3.1: RFECV with Tuned XGBoost
  (starting from 113 features)

RFECV complete!  30 features selected from 113 candidates.
Selected features:
  • home_fga
  • home_fg2a
  • home_trb
  • home_stl
  • home_pf
  • home_drtg
  • home_trb_pct
  • home_stl_pct
  • home_blk_pct
  • home_elo
  • away_fg
  • away_fg3a
  • away_ft
  • away_fta
  • away_ft_pct
  • away_blk
  • away_ftr
  • away_3ptar
  • away_ast_pct
  • away_tov_pct
  • away_ft_rate
  • away_fg_pct_away_avg
  • away_drtg_away_avg
  • away_elo
  • home_moneyline_implied_prob
  • away_moneyline_implied_prob
  • prob_diff
  • prob_ratio
  • elo_diff
  • elo_ratio

  Phase 4: Evaluating Base Models for Ensemble Weights
  xgboost      CV log-loss: 0.59811
  lightgbm     CV log-loss: 0.59900
  catboost     CV log-loss: 0.59742

  Voting weights (log-loss-derived): {'xgboost': 0.3334, 'lightgbm': 0.3329, 'catboost': 0.3338}

  Phase 5.1: Calibrating Ensemble (isotonic / TimeSeriesSplit)
  F